In [ ]:
!pip install -q openai

In [ ]:
from IPython.display import Image, Markdown, Video

def print_generation(path_to_thumbnail, path_to_video):
    display(Markdown("### Thumbnail"))
    display(Image(path_to_thumbnail, embed=True))
    display(Markdown("### Video"))
    display(Video(path_to_video, embed=True))

In [ ]:
from google.colab import userdata
from openai import OpenAI

api_key = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=api_key)

In [ ]:
import time

def generate(*args, **kwargs):
    video = client.videos.create(*args, **kwargs)

    for i in range(1000):
        if video.status == "completed":
            return video
        if video.status == "failed":
            raise RuntimeError(f"Generation failed: {video['error']['message']}")

        time.sleep(2)
        video = client.videos.retrieve(video_id=video.id)

    raise RuntimeError("Video generation couldn't finish in time.")

In [ ]:
from pathlib import Path

def download_thumbnail(video, path_to_file: Path):
    thumbnail_bytes = client.videos.download_content(video_id=video.id, variant="thumbnail")
    with path_to_file.open(mode="wb") as file:
        file.write(thumbnail_bytes.content)


def download_video(video, path_to_file: Path):
    video_bytes = client.videos.download_content(video_id=video.id, variant="video")
    with path_to_file.open(mode="wb") as file:
        file.write(video_bytes.content)

## Text to Video

In [ ]:
funny_cat_prompt = "Generate a short and funny video of a cat that is playing with its tail, while laying on a mat."

In [ ]:
funny_cat_generation = generate(
    prompt=funny_cat_prompt,
    model="sora-2",
    seconds="4",
    size="1280x720"
)

In [ ]:
download_thumbnail(funny_cat_generation, Path("/content/funny_cat_thumbnail.png"))

In [ ]:
download_video(funny_cat_generation, Path("/content/funny_cat_video.mp4"))

In [ ]:
print_generation("/content/funny_cat_thumbnail.png", "/content/funny_cat_video.mp4")

## Image to Video

In [ ]:
walking_cat_prompt = "Extend this schene with a cat walking on the kitchen counter."

In [ ]:
walking_cat_generation = generate(
    prompt=walking_cat_prompt,
    model="sora-2",
    seconds="4",
    size="1280x720",

    # NOTE: The reference image's dimensions must match the size of the video.
    input_reference=Path("/content/kitchen.jpg").open(mode="rb")
)

In [ ]:
download_thumbnail(walking_cat_generation, Path("/content/walking_cat_thumbnail.png"))

In [ ]:
download_video(walking_cat_generation, Path("/content/walking_cat_video.mp4"))

In [ ]:
print_generation("/content/walking_cat_thumbnail.png", "/content/walking_cat_video.mp4")